In [3]:
!pip install transformers datasets torch accelerate

In [5]:
#########################
### CONSTANTES KAGGLE ###
#########################

FUNCTIONS_PATH = "/kaggle/input/datasets/frmorales/functions"

SOLIDIFY_DATASET_PATH = "/kaggle/input/datasets/frmorales/solidify-benchmark/results"

SLITHER_DATASET_PATH = "/kaggle/input/datasets/grawatschp/slither-audited-smart-contracts"

WILD_DATASET_PATH = "/kaggle/input/datasets/tranduongminhdai/smartbug-dataset/smartbugs_wild.csv"

In [ ]:
if FUNCTIONS_PATH not in sys.path:
    sys.path.append(FUNCTIONS_PATH)

try:
    from deduplication import deduplicate_df
    from tokenization import truncate_by_token_length

    from preprocessing import preprocess_source

    print("✅ Módulo cargado correctamente")
except ImportError as e:
    print(f"❌ Error al importar: {e}")
    print("Contenido del dataset:", os.listdir(FUNCTIONS_PATH))

✅ Módulo cargado correctamente


# SOLIDIFY-BENCHMARK DATASET

In [7]:
def build_solidify_dataset(base_path):
    raw_data = {}

    herramientas = ["Manticore", "Mythril", "Oyente", "Securify", "Slither", "SmartCheck"]

    carpetas_vulnerabilidades = [
        "Overflow-Underflow",
        "Re-entrancy",
        "TOD",
        "Timestamp-Dependency",
        "Unchecked-Send",
        "Unhandled-Exceptions",
        "tx.origin",
    ]

    for tool in herramientas:
        tool_path = os.path.join(base_path, tool, "analyzed_buggy_contracts")

        if not os.path.exists(tool_path):
            continue

        for folder_name in carpetas_vulnerabilidades:
            cat_path = os.path.join(tool_path, folder_name)
            if not os.path.exists(cat_path):
                continue

            for file_name in os.listdir(cat_path):
                if file_name.endswith(".sol"):
                    sol_path = os.path.join(cat_path, file_name)

                    try:
                        with open(sol_path, "r", encoding="utf-8", errors="ignore") as f:
                            code = f.read().strip()
                    except:
                        continue

                    if not code:
                        continue

                    if code not in raw_data:
                        raw_data[code] = set()

                    raw_data[code].add(folder_name)

    final_list = []
    for code, categories_set in raw_data.items():
        final_list.append({"source_code": code, "categories": list(categories_set)})

    df_final = pd.DataFrame(final_list)
    return df_final

In [8]:
df_solidifi = build_solidify_dataset(SOLIDIFY_DATASET_PATH)

In [9]:
print(f"Cantidad total de contratos: {len(df_solidifi)}")

Cantidad total de contratos: 679


In [10]:
def preprocess_solidify(df):
    mapping = {
        "Re-entrancy": "Re-entrancy",
        "Timestamp-Dependency": "Timestamp-Dependency",
        "Unhandled-Exceptions": "Unhandled-Exception",
        "Unchecked-Send": "Unhandled-Exception",
        "tx.origin": "tx.origin",
    }

    def map_solidify_labels(cat_list):
        found = set()
        for cat in cat_list:
            if cat in mapping:
                found.add(mapping[cat])

        if not found:
            return {"clean"}
        return found

    df["final_labels"] = df["categories"].apply(map_solidify_labels)

    target_cols = [
        "Re-entrancy",
        "Timestamp-Dependency",
        "Unhandled-Exception",
        "tx.origin",
        "clean",
    ]
    for col in target_cols:
        df[col] = df["final_labels"].apply(lambda x: 1 if col in x else 0)

    return df

In [11]:
df_solidifi_final = preprocess_solidify(df_solidifi)

In [12]:
print("\n" + "=" * 50)
print("RESULTADOS: SOLIDIFY")
print("=" * 50)

vulnerability_cols = ["Re-entrancy", "Timestamp-Dependency", "Unhandled-Exception", "tx.origin"]
conteo = df_solidifi_final[vulnerability_cols + ["clean"]].sum().sort_values(ascending=False)

for label, count in conteo.items():
    print(f"{label.ljust(25)}: {int(count)}")

print("-" * 50)
print(f"Total contratos: {len(df_solidifi_final)}")
print("=" * 50)


RESULTADOS: SOLIDIFY
clean                    : 236
Unhandled-Exception      : 148
Re-entrancy              : 147
Timestamp-Dependency     : 99
tx.origin                : 49
--------------------------------------------------
Total contratos: 679


In [13]:
def save_solidify(df, filename="solidify_dataset.csv"):
    df_export = df.copy()

    df_export["contract_id"] = [f"SOLIDIFI_{i + 1:04d}" for i in range(len(df))]

    df_export["source_len"] = df_export["source_code"].str.len()

    df_export["dataset_origin"] = "solidify_benchmark"

    target_vulnerabilities = [
        "Re-entrancy",
        "Timestamp-Dependency",
        "Unhandled-Exception",
        "tx.origin",
    ]

    for col in target_vulnerabilities:
        if col not in df_export.columns:
            df_export[col] = 0

    column_order = [
        "contract_id",
        "source_len",
        "dataset_origin",
        "Re-entrancy",
        "Timestamp-Dependency",
        "Unhandled-Exception",
        "tx.origin",
        "source_code",
    ]

    df_final_export = df_export[column_order]

    df_final_export.to_csv(filename, index=False, encoding="utf-8")

In [14]:
save_solidify(df_solidifi_final)

# SLITHER AUDITED SMART CONTRACTS DATASET

In [15]:
SLITHER_DATASET_PATH = "/kaggle/input/datasets/grawatschp/slither-audited-smart-contracts"

parquet_files = glob.glob(f"{SLITHER_DATASET_PATH}/**/*.parquet", recursive=True)

print(f"Se encontraron {len(parquet_files)} archivos Parquet.")

df_list = [pd.read_parquet(file) for file in parquet_files]
df_slither = pd.concat(df_list, ignore_index=True)

print(f"Cantidad total de contratos: {len(df_slither)}")

Se encontraron 3 archivos Parquet.
Cantidad total de contratos: 14134


In [16]:
import numpy as np
import pandas as pd


def preprocess_slither_dataset(df):
    def extract_specific_vulnerabilities(row):
        label_list = row["slither"]
        code = str(row["source_code"]).lower()

        if label_list is None or len(label_list) == 0:
            return {"clean"}

        ids = set(label_list)
        if ids == {4}:
            return {"clean"}

        found = set()

        # 1. Re-entrancy (ID 3)
        if 3 in ids:
            found.add("Re-entrancy")

        # 2. Unhandled-Exception (ID 5)
        if 5 in ids:
            found.add("Unhandled-Exception")

        # 3. tx.origin (ID 0 o 7 + Validación de palabra clave)
        if (0 in ids or 7 in ids) and ("tx.origin" in code):
            found.add("tx.origin")

        # 4. Timestamp-Dependency (ID 2 o 6 + Validación de palabra clave)
        if (2 in ids or 6 in ids) and ("timestamp" in code or "block.timestamp" in code):
            found.add("Timestamp-Dependency")

        if not found:
            return {"clean"}
        return found

    df["final_labels"] = df.apply(extract_specific_vulnerabilities, axis=1)

    target_cols = [
        "Re-entrancy",
        "Timestamp-Dependency",
        "Unhandled-Exception",
        "tx.origin",
        "clean",
    ]
    for col in target_cols:
        df[col] = df["final_labels"].apply(lambda x: 1 if col in x else 0)

    return df

In [17]:
df_slither = preprocess_slither_dataset(df_slither)

In [18]:
print("\n" + "=" * 50)
print("RESULTADOS: SLITHER AUDITED DATASET")
print("=" * 50)

vulnerability_cols = ["Re-entrancy", "Timestamp-Dependency", "Unhandled-Exception", "tx.origin"]
conteo = df_slither[vulnerability_cols + ["clean"]].sum().sort_values(ascending=False)

for label, count in conteo.items():
    print(f"{label.ljust(25)}: {int(count)}")

print("-" * 50)
print(f"Total de contratos procesados: {len(df_slither)}")
print("=" * 50)


RESULTADOS: SLITHER AUDITED DATASET
clean                    : 8915
Unhandled-Exception      : 3480
Timestamp-Dependency     : 3310
Re-entrancy              : 2426
tx.origin                : 569
--------------------------------------------------
Total de contratos procesados: 14134


In [19]:
def save_slither(df, filename="slither_dataset.csv"):
    df_export = df.copy()

    df_export["contract_id"] = [f"SLITHER_{i + 1:06d}" for i in range(len(df))]

    df_export["source_len"] = df_export["source_code"].str.len()

    df_export["dataset_origin"] = "slither_audited"

    target_vulnerabilities = [
        "Re-entrancy",
        "Timestamp-Dependency",
        "Unhandled-Exception",
        "tx.origin",
    ]

    for col in target_vulnerabilities:
        if col not in df_export.columns:
            df_export[col] = 0

    column_order = [
        "contract_id",
        "source_len",
        "dataset_origin",
        "Re-entrancy",
        "Timestamp-Dependency",
        "Unhandled-Exception",
        "tx.origin",
        "source_code",
    ]

    df_final_export = df_export[column_order]

    df_final_export.to_csv(filename, index=False, encoding="utf-8")


def save_slither_vulnerable_only(df, filename="slither_dataset.csv"):
    df_export = df.copy()

    # 1. Mapeo de IDs de Slither a tus columnas objetivo
    # ID 3: Reentrancy
    # ID 5: Unhandled-Exception (unchecked_low_calls)
    # ID 0: tx.origin (access_control en este dataset)
    # ID 2: Timestamp-Dependency (mapeado desde 'other' por Slither)

    df_export["Re-entrancy"] = df_export["slither"].apply(lambda x: 1 if 3 in x else 0)
    df_export["Unhandled-Exception"] = df_export["slither"].apply(lambda x: 1 if 5 in x else 0)
    df_export["tx.origin"] = df_export["slither"].apply(lambda x: 1 if 0 in x else 0)
    df_export["Timestamp-Dependency"] = df_export["slither"].apply(lambda x: 1 if 2 in x else 0)

    # 2. Filtrar: Solo nos quedamos con contratos que tengan AL MENOS una vulnerabilidad
    target_vulnerabilities = [
        "Re-entrancy",
        "Timestamp-Dependency",
        "Unhandled-Exception",
        "tx.origin",
    ]

    # Sumamos las filas de las columnas objetivo; si la suma es > 0, es vulnerable
    mask_vulnerable = df_export[target_vulnerabilities].sum(axis=1) > 0
    df_final = df_export[mask_vulnerable].copy()

    if len(df_final) == 0:
        print("⚠️ ¡Ojo! El filtro resultó en 0 contratos. Revisa los IDs de Slither.")
        return

    # 3. Metadatos y formato
    df_final["contract_id"] = [f"SLITHER_{i + 1:06d}" for i in range(len(df_final))]
    df_final["source_len"] = df_final["source_code"].str.len()
    df_final["dataset_origin"] = "slither_audited"

    column_order = [
        "contract_id",
        "source_len",
        "dataset_origin",
        "Re-entrancy",
        "Timestamp-Dependency",
        "Unhandled-Exception",
        "tx.origin",
        "source_code",
    ]

    df_final_export = df_final[column_order]

    # Guardar
    df_final_export.to_csv(filename, index=False, encoding="utf-8")
    print(f"✅ Guardado con éxito. Se exportaron {len(df_final_export)} contratos vulnerables.")
    print(df_final_export[target_vulnerabilities].sum())

In [20]:
save_slither_vulnerable_only(df_slither)

✅ Guardado con éxito. Se exportaron 5758 contratos vulnerables.
Re-entrancy             2426
Timestamp-Dependency    4098
Unhandled-Exception     3480
tx.origin               1477
dtype: int64


# SMARTBUGS WILD DATASET

In [21]:
df_wild = pd.read_csv(WILD_DATASET_PATH)

In [22]:
print(f"Cantidad total de contratos: {len(df_wild)}")

Cantidad total de contratos: 47451


In [23]:
import ast
from collections import Counter


def extract_all_vulnerabilities(tools_dict):
    vuln_list = []

    if isinstance(tools_dict, str):
        try:
            tools_dict = ast.literal_eval(tools_dict)
        except:
            return []

    if not isinstance(tools_dict, dict):
        return []

    for tool_name, tool_data in tools_dict.items():
        if isinstance(tool_data, dict):
            vulnerabilities = tool_data.get("vulnerabilities", [])
            if vulnerabilities is None:
                continue
            for v in vulnerabilities:
                vuln_list.append(str(v).lower())
    return vuln_list


all_vulns_flat = []
for tools_entry in df_wild["tools"]:
    all_vulns_flat.extend(extract_all_vulnerabilities(tools_entry))

counts = Counter(all_vulns_flat)

print(f"{'Vulnerabilidad Detectada':<45} | {'Apariciones':<10}")
print("-" * 60)

for vuln, count in counts.most_common():
    print(f"{vuln:<45} | {count:<10}")

invalid_entries = sum(1 for x in df_wild["tools"] if not isinstance(x, (dict, str)))
print("-" * 60)
print(f"Entradas inválidas o nulas saltadas: {invalid_entries}")

Vulnerabilidad Detectada                      | Apariciones
------------------------------------------------------------
integer overflow.                             | 30992     
integer underflow.                            | 23646     
integer overflow                              | 18346     
overflow_bugs                                 | 11690     
solidity_gas_limit_in_loops                   | 11505     
exception state                               | 10832     
solidity_functions_returns_type_and_no_return | 9786      
message call to external contract             | 8454      
reentrancy-benign                             | 7647      
unused-return                                 | 7523      
solidity_locked_money                         | 6436      
reentrancy-no-eth                             | 6339      
todamount                                     | 5614      
underflow_bugs                                | 5524      
low-level-calls                               | 5133 

In [24]:
import pandas as pd


def preprocess_wild_dataset(df):
    mapping = {
        # 1. Re-entrancy
        "reentrancy-benign": "Re-entrancy",
        "reentrancy-no-eth": "Re-entrancy",
        "reentrancy-eth": "Re-entrancy",
        "reentrancy_bug": "Re-entrancy",
        "re-entrancy vulnerability.": "Re-entrancy",
        "potential reentrancy vulnerability": "Re-entrancy",
        "reentrancy multi-million ether bug": "Re-entrancy",
        "message call to external contract": "Re-entrancy",
        # 2. Timestamp-Dependency
        "timestamp": "Timestamp-Dependency",
        "time_dependency_bug": "Timestamp-Dependency",
        "timestamp dependency.": "Timestamp-Dependency",
        "dependence on predictable environment variable": "Timestamp-Dependency",
        "warning timestamp instruction used": "Timestamp-Dependency",
        "solidity_exact_time": "Timestamp-Dependency",
        # 3. Unhandled-Exception
        "unhandledexception": "Unhandled-Exception",
        "unchecked call return value": "Unhandled-Exception",
        "solidity_unchecked_call": "Unhandled-Exception",
        "callstack depth attack vulnerability.": "Unhandled-Exception",
        "callstack_bug": "Unhandled-Exception",
        "exception state": "Unhandled-Exception",
        # 4. tx.origin
        "use of tx.origin": "tx.origin",
        "solidity_tx_origin": "tx.origin",
        "tx-origin": "tx.origin",
        "warning origin instruction used": "tx.origin",
    }

    def process_tools_consenso(tools_entry):
        """
        Analiza el diccionario de tools y retorna:
        1. El set de categorías únicas encontradas (para One-Hot).
        2. Un string descriptivo de 'Herramienta: Error'.
        """
        tools_dict = {}
        if isinstance(tools_entry, str):
            try:
                tools_dict = ast.literal_eval(tools_entry)
            except:
                return set(["clean"]), ""
        elif isinstance(tools_entry, dict):
            tools_dict = tools_entry
        else:
            return set(["clean"]), ""

        found_categories = set()
        detecciones = []

        for tool_name, tool_data in tools_dict.items():
            if not isinstance(tool_data, dict):
                continue

            vulns = tool_data.get("vulnerabilities", [])
            if vulns is None:
                continue

            for v in vulns:
                v_clean = str(v).lower().strip()
                if v_clean in mapping:
                    categoria = mapping[v_clean]
                    found_categories.add(categoria)
                    detecciones.append(f"{tool_name}: {categoria}")

        if not found_categories:
            return set(["clean"]), ""

        detecciones_str = "; ".join(sorted(list(set(detecciones))))
        return found_categories, detecciones_str

    results = df["tools"].apply(process_tools_consenso)
    df["final_labels"] = results.apply(lambda x: x[0])
    df["detection_details"] = results.apply(lambda x: x[1])

    target_cols = [
        "Re-entrancy",
        "Timestamp-Dependency",
        "Unhandled-Exception",
        "tx.origin",
        "clean",
    ]
    for col in target_cols:
        df[col] = df["final_labels"].apply(lambda x: 1 if col in x else 0)

    return df


df_wild_processed = preprocess_wild_dataset(df_wild)

In [25]:
print("=" * 50)
print("RESULTADOS: SMARTBUGS WILD")
print("=" * 50)
print(f"Total Contratos: {len(df_wild_processed)}")
print(f"Total Clean:    {df_wild_processed['clean'].sum()}")
print("-" * 50)
for cat in ["Re-entrancy", "Timestamp-Dependency", "Unhandled-Exception", "tx.origin"]:
    print(f"{cat:<25}: {df_wild_processed[cat].sum()}")
print("=" * 50)

RESULTADOS: SMARTBUGS WILD
Total Contratos: 47451
Total Clean:    23215
--------------------------------------------------
Re-entrancy              : 13397
Timestamp-Dependency     : 4267
Unhandled-Exception      : 12969
tx.origin                : 810


In [26]:
import pandas as pd

# ── Hiperparámetros ────────────────────────────────────────────────────────────
T = 0.8
MU_C = 0.1
EQUAL_RATE = 20.0

# ── Matriz de Capacidad de Detección (MCD) ────────────────────────────────────
MCD = {
    "honeybadger": {"reentrancy": 0, "time_manipulation": 0, "unchecked_low_calls": 0, "other": 67},
    "maian": {"reentrancy": 0, "time_manipulation": 0, "unchecked_low_calls": 0, "other": 0},
    "manticore": {"reentrancy": 25, "time_manipulation": 20, "unchecked_low_calls": 17, "other": 0},
    "mythril": {"reentrancy": 62, "time_manipulation": 0, "unchecked_low_calls": 42, "other": 0},
    "osiris": {"reentrancy": 62, "time_manipulation": 0, "unchecked_low_calls": 0, "other": 0},
    "oyente": {"reentrancy": 62, "time_manipulation": 0, "unchecked_low_calls": 0, "other": 0},
    "securify": {"reentrancy": 62, "time_manipulation": 0, "unchecked_low_calls": 25, "other": 0},
    "slither": {"reentrancy": 88, "time_manipulation": 40, "unchecked_low_calls": 33, "other": 100},
    "smartcheck": {
        "reentrancy": 62,
        "time_manipulation": 20,
        "unchecked_low_calls": 33,
        "other": 0,
    },
}

TOOLS = list(MCD.keys())
N_TOOLS = len(TOOLS)

VULN_MAP = {
    "Re-entrancy": "reentrancy",
    "Timestamp-Dependency": "time_manipulation",
    "Unhandled-Exception": "unchecked_low_calls",
    "tx.origin": "other",
}

VULN_CATS = list(VULN_MAP.keys())


# ── Helpers de Procesamiento ──────────────────────────────────────────────────
def _safe_get_tools_dict(tools_entry):
    """Convierte la columna tools a diccionario de forma segura."""
    if isinstance(tools_entry, dict):
        return tools_entry
    if isinstance(tools_entry, str):
        try:
            return ast.literal_eval(tools_entry)
        except:
            return {}
    return {}


def _is_strictly_clean(tools_dict):
    """
    Verifica si ABSOLUTAMENTE NINGUNA herramienta detectó NADA
    (revisando el campo vulnerabilities de cada una).
    """
    if not tools_dict:
        return True

    for tool in tools_dict:
        # Si la herramienta tiene una lista de vulnerabilidades y no está vacía
        vulns = tools_dict[tool].get("vulnerabilities")
        if vulns and len(vulns) > 0:
            return False
    return True


# ── Funciones principales ──────────────────────────────────────────────────────
def compute_scores(details_str: str, tools_dict: dict, t: float = T) -> dict:
    scores = {cat: 0.0 for cat in VULN_CATS}

    # 1. Calcular scores para las vulnerabilidades detectadas
    if pd.notna(details_str) and str(details_str).strip() != "":
        detecciones = [d.strip().split(": ") for d in str(details_str).split(";")]
        for item in detecciones:
            if len(item) != 2:
                continue
            tool_name, vuln_name = item[0].lower(), item[1]

            if tool_name in MCD and vuln_name in VULN_MAP:
                mcd_key = VULN_MAP[vuln_name]
                rate = MCD[tool_name][mcd_key]
                scores[vuln_name] += np.exp(rate / (100.0 * t))

    # 2. Normalizar scores de vulnerabilidades
    for cat in VULN_CATS:
        scores[cat] = scores[cat] / N_TOOLS

    # 3. Lógica CLEAN ESTRICTA
    # Es clean (1.0) solo si pasó la verificación de _is_strictly_clean
    scores["clean"] = 1.0 if _is_strictly_clean(tools_dict) else 0.0

    return scores


def assign_labels(details_str: str, tools_dict: dict, mu_c: float = MU_C, t: float = T) -> set | None:
    scores = compute_scores(details_str, tools_dict, t=t)

    # Aceptamos labels que superen el umbral
    accepted = {cat for cat, score in scores.items() if score >= mu_c}

    if not accepted:
        return None

    vuln_labels = accepted - {"clean"}
    # Prioridad: Si hay vulnerabilidades aceptadas por MCD, devolvemos esas.
    # Si no hay ninguna, pero 'clean' es 1.0, devolvemos {'clean'}.
    return vuln_labels if vuln_labels else ({"clean"} if scores["clean"] == 1.0 else None)


def apply_mcd_filter(
    df: pd.DataFrame,
    details_col: str = "detection_details",
    tools_col: str = "tools",
    t: float = T,
    mu_c: float = MU_C,
) -> pd.DataFrame:
    df = df.copy()

    # Procesar tools como diccionario una sola vez para ganar eficiencia
    df["tools_parsed"] = df[tools_col].apply(_safe_get_tools_dict)

    # Aplicar cálculos pasando ambos datos
    df["scores"] = df.apply(lambda row: compute_scores(row[details_col], row["tools_parsed"], t=t), axis=1)
    df["accepted_labels"] = df.apply(
        lambda row: assign_labels(row[details_col], row["tools_parsed"], mu_c=mu_c, t=t), axis=1
    )

    # Filtrado final
    n_before = len(df)
    df = df[df["accepted_labels"].notna()].reset_index(drop=True)
    n_discarded = n_before - len(df)

    print(f"Hiperparámetros usados: T={t}, μ_c={mu_c}")
    print(f"Contratos descartados (sin consenso suficiente): {n_discarded}")
    print(f"Contratos restantes: {len(df)}")

    return df.drop(columns=["tools_parsed"])  # Limpiamos la columna temporal

In [27]:
df_verified = df_wild_processed.copy()

print(f"Hiperparámetros: T={T}, μ_c={MU_C}, EQUAL_RATE={EQUAL_RATE}")
print()
df_verified = apply_mcd_filter(df_verified, t=1.2, mu_c=0.3)

Hiperparámetros: T=0.8, μ_c=0.1, EQUAL_RATE=20.0

Hiperparámetros usados: T=1.2, μ_c=0.3
Contratos descartados (sin consenso suficiente): 39441
Contratos restantes: 8010


In [28]:
print("=" * 50)
print("RESULTADOS: SMARTBUGS WILD")
print("=" * 50)
print(f"Total Contratos: {len(df_verified)}")
print(f"Total Clean:    {df_verified['clean'].sum()}")
print("-" * 50)
for cat in ["Re-entrancy", "Timestamp-Dependency", "Unhandled-Exception", "tx.origin"]:
    print(f"{cat:<25}: {df_verified[cat].sum()}")
print("=" * 50)

RESULTADOS: SMARTBUGS WILD
Total Contratos: 8010
Total Clean:    2862
--------------------------------------------------
Re-entrancy              : 4647
Timestamp-Dependency     : 812
Unhandled-Exception      : 1698
tx.origin                : 169


In [29]:
import pandas as pd


def save_wild(df, filename="wild_dataset.csv"):
    df_export = df.copy()

    df_export["contract_id"] = [f"MCD_WILD_{i + 1:06d}" for i in range(len(df))]

    df_export["source_len"] = df_export["source_code"].str.len()

    df_export["dataset_origin"] = "smartbugs_wild"

    target_vulnerabilities = [
        "Re-entrancy",
        "Timestamp-Dependency",
        "Unhandled-Exception",
        "tx.origin",
    ]

    for col in target_vulnerabilities:
        if col not in df_export.columns:
            df_export[col] = 0
        else:
            df_export[col] = df_export[col].astype(int)

    column_order = [
        "contract_id",
        "source_len",
        "dataset_origin",
        "Re-entrancy",
        "Timestamp-Dependency",
        "Unhandled-Exception",
        "tx.origin",
        "source_code",
    ]

    df_final_export = df_export[column_order]
    df_final_export.to_csv(filename, index=False, encoding="utf-8")

    print(f"✅ Dataset guardado con éxito: {filename}")
    print(f"📊 Filas: {len(df_final_export)} | Columnas: {list(df_final_export.columns)}")


save_wild(df_verified)

✅ Dataset guardado con éxito: wild_dataset.csv
📊 Filas: 8010 | Columnas: ['contract_id', 'source_len', 'dataset_origin', 'Re-entrancy', 'Timestamp-Dependency', 'Unhandled-Exception', 'tx.origin', 'source_code']


# LOAD DATASETS AND BUILD TRAIN AND TEST SUITE

In [30]:
import pandas as pd

# 1. Cargar los datasets
SOLIDIFY_PATH = "/kaggle/working/solidify_dataset.csv"
SLITHER_PATH = "/kaggle/working/slither_dataset.csv"
WILD_PATH = "/kaggle/working/wild_dataset.csv"

# SOLIDIFY_PATH = "/kaggle/input/datasets/frmorales/try-dataset/solidify_dataset.csv"
# SLITHER_PATH  = "/kaggle/input/datasets/frmorales/try-dataset/slither_dataset.csv"
# WILD_PATH     = "/kaggle/input/datasets/frmorales/try-dataset/wild_dataset.csv"

solidify_df = pd.read_csv(SOLIDIFY_PATH)
slither_df = pd.read_csv(SLITHER_PATH)
wild_df = pd.read_csv(WILD_PATH)

# 2. Concatenar Slither y Wild para formar el set de Entrenamiento (Train)
# Usamos ignore_index=True para que no se repitan los índices originales
train_df = pd.concat([slither_df, wild_df], ignore_index=True)

# 3. El set de Test es Solidify
test_df = solidify_df.copy()

# 4. Mostrar métricas de los grupos
print("📊 Resumen de Datasets")
print("-" * 40)
print(f"🔹 Slither (Audited): {len(slither_df)} entradas")
print(f"🔹 Wild (MCD Filtered): {len(wild_df)} entradas")
print(f"🔸 Total TRAIN: {train_df.shape[0]} entradas")
print(f"🔸 Total TEST (Solidify): {test_df.shape[0]} entradas")
print("-" * 40)

# 5. Verificación de consistencia
print(f"✅ Columnas train: {list(train_df.columns)}")
print(f"✅ Columnas test:  {list(test_df.columns)}")

📊 Resumen de Datasets
----------------------------------------
🔹 Slither (Audited): 5758 entradas
🔹 Wild (MCD Filtered): 8010 entradas
🔸 Total TRAIN: 13768 entradas
🔸 Total TEST (Solidify): 679 entradas
----------------------------------------
✅ Columnas train: ['contract_id', 'source_len', 'dataset_origin', 'Re-entrancy', 'Timestamp-Dependency', 'Unhandled-Exception', 'tx.origin', 'source_code']
✅ Columnas test:  ['contract_id', 'source_len', 'dataset_origin', 'Re-entrancy', 'Timestamp-Dependency', 'Unhandled-Exception', 'tx.origin', 'source_code']


In [31]:
#####################
### NORMALIZACIÓN ###
#####################

train_df["source_code"] = train_df["source_code"].astype(str).replace("nan", "")
test_df["source_code"] = test_df["source_code"].astype(str).replace("nan", "")

train_df["source_code"] = train_df["source_code"].apply(preprocess_source)
test_df["source_code"] = test_df["source_code"].apply(preprocess_source)

In [32]:
#########################################################
### FUNCIÓN: MINIFICACIÓN DE CÓDIGO SOLIDITY ###
#########################################################

"""
TODO
Elimina comentarios para identificar correctamente el último contrato.
Armar otra opción donde no sea necesario eliminar comentarios ya que pueden poseer información valiosa.
"""


def clean_extract_and_minify(code: str) -> str:
    """
    Realiza una limpieza del código fuente para eliminar código de librerias.

    Objetivos principales:
    1. Enfoque de Lógica: Extrae solo el último contrato del archivo (generalmente
       el contrato principal).
    2. Optimización de Tokens: La minificación agresiva permite que una mayor
       cantidad de lógica de control quepa dentro del límite de 512 tokens de CodeBERT.

    Pasos:
    - Regex 1 & 2: Remoción de comentarios multilínea (/* */) y unilínea (//).
    - Regex 3: Identifica todos los 'contract Name' y recorta el string desde el inicio del último.

    Args:
        code (str): Código fuente original de Solidity.

    Returns:
        str: Código minificado.
    """
    if not isinstance(code, str):
        return ""

    code = re.sub(r"/\*.*?\*/", "", code, flags=re.DOTALL)
    code = re.sub(r"//.*", "", code)

    matches = list(re.finditer(r"\bcontract\s+(\w+)", code))
    if matches:
        last_contract_start = matches[-1].start()
        code = code[last_contract_start:]

    return code.strip()

In [33]:
train_df_min = train_df.copy()
test_df_min = test_df.copy()

train_df_min["source_code"] = train_df_min["source_code"].apply(clean_extract_and_minify)
test_df_min["source_code"] = test_df_min["source_code"].apply(clean_extract_and_minify)

In [34]:
train_df_min = deduplicate_df(train_df_min)
test_df_min = deduplicate_df(test_df_min)

Contratos antes de deduplicación : 13768
Duplicados eliminados            : 1964
Contratos restantes              : 11804
Contratos antes de deduplicación : 679
Duplicados eliminados            : 42
Contratos restantes              : 637


In [35]:
n_before_train = len(train_df_min)
n_before_test = len(test_df_min)
train_df_min = train_df_min[train_df_min["source_code"].apply(lambda x: isinstance(x, str) and len(x) > 0)].reset_index(
    drop=True
)
test_df_min = test_df_min[test_df_min["source_code"].apply(lambda x: isinstance(x, str) and len(x) > 0)].reset_index(
    drop=True
)

print(f"Contratos train con source_code inválido removidos: {n_before_train - len(train_df_min)}")
print(f"Contratos test con source_code inválido removidos: {n_before_test - len(test_df_min)}")

Contratos train con source_code inválido removidos: 1
Contratos test con source_code inválido removidos: 0


In [36]:
train_df_min_token = train_df_min.copy()
test_df_min_token = test_df_min.copy()
tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")

train_df_min_token = truncate_by_token_length(train_df_min_token, tokenizer, max_tokens=512)
test_df_min_token = truncate_by_token_length(test_df_min_token, tokenizer, max_tokens=512)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1845 > 512). Running this sequence through the model will result in indexing errors


Calculando longitudes de tokenización...
Contratos truncados (> 512 tokens): 6024  (51.0%)
Total contratos                            : 11803

count    11803.000000
mean      1054.411167
std       1564.025908
min          9.000000
25%        330.000000
50%        536.000000
75%       1238.500000
max      28753.000000
Calculando longitudes de tokenización...
Contratos truncados (> 512 tokens): 543  (85.2%)
Total contratos                            : 637

count     637.000000
mean     1807.351648
std      1439.366112
min        61.000000
25%       791.000000
50%      1494.000000
75%      2483.000000
max      9512.000000


In [37]:
# Lista de tus 4 vulnerabilidades objetivo
target_vuls = ["Re-entrancy", "Timestamp-Dependency", "Unhandled-Exception", "tx.origin"]


def apply_clean_logic(df):
    """
    Inyecta la columna 'clean' basada en la ausencia de las 4 vulnerabilidades.
    Asegura que los valores sean enteros.
    """
    d = df.copy()

    # 1. Asegurar que las 4 vuls sean numéricas (por si hay NaNs del concat)
    for v in target_vuls:
        if v in d.columns:
            d[v] = pd.to_numeric(d[v], errors="coerce").fillna(0).astype(int)
        else:
            d[v] = 0

    # 2. Definir 'clean': 1 si la suma de las vuls es 0, de lo contrario 0
    d["clean"] = (d[target_vuls].sum(axis=1) == 0).astype(int)

    return d


# Aplicamos la lógica a tus datasets procesados
train_df_min_token = apply_clean_logic(train_df_min_token)
test_df_min_token = apply_clean_logic(test_df_min_token)

# --- AHORA SÍ CORREMOS LA DISTRIBUCIÓN ---


def plot_vulnerability_distribution(df, title="Distribución"):
    vulnerabilities = target_vuls + ["clean"]
    counts = df[vulnerabilities].sum().sort_values(ascending=False)

    print(f"\n📊 {title}")
    print("-" * 35)
    for vuln, count in counts.items():
        percentage = (count / len(df)) * 100
        print(f"{vuln:<22}: {int(count):>6} ({percentage:>6.2f}%)")
    print(f"{'TOTAL CONTRATOS':<22}: {len(df):>6}")
    print("-" * 35)


plot_vulnerability_distribution(train_df_min_token, "TRAIN (Slither + Wild)")
plot_vulnerability_distribution(test_df_min_token, "TEST (Solidify)")


📊 TRAIN (Slither + Wild)
-----------------------------------
Re-entrancy           :   5997 ( 50.81%)
Unhandled-Exception   :   4069 ( 34.47%)
Timestamp-Dependency  :   3830 ( 32.45%)
clean                 :   2615 ( 22.16%)
tx.origin             :   1232 ( 10.44%)
TOTAL CONTRATOS       :  11803
-----------------------------------

📊 TEST (Solidify)
-----------------------------------
clean                 :    196 ( 30.77%)
Unhandled-Exception   :    148 ( 23.23%)
Re-entrancy           :    146 ( 22.92%)
Timestamp-Dependency  :     98 ( 15.38%)
tx.origin             :     49 (  7.69%)
TOTAL CONTRATOS       :    637
-----------------------------------


In [ ]:
os.environ["HF_TOKEN"] = ""
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["TRANSFORMERS_CACHE"] = os.path.join(os.getcwd(), "cache")
os.environ["HF_HOME"] = os.path.join(os.getcwd(), "cache")

In [39]:
import torch
import torch.nn as nn
from datasets import Dataset as HFDataset
from sklearn.metrics import accuracy_score, f1_score, recall_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

# --- CONFIGURACIÓN ---
MAX_LEN = 512
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-6
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Definimos las clases exactas que calculamos antes
VULN_CLASSES = ["Re-entrancy", "Timestamp-Dependency", "Unhandled-Exception", "tx.origin", "clean"]
NUM_LABELS = len(VULN_CLASSES)

# 1. Preparación de Labels (Ya no necesitamos parse_labels, usamos las columnas One-Hot)
# Convertimos las columnas a un array de numpy (float32 para la loss function)
train_labels = train_df_min_token[VULN_CLASSES].values.astype(np.float32)
test_labels = test_df_min_token[VULN_CLASSES].values.astype(np.float32)

# 2. Carga del Tokenizador y Modelo
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")

model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/codebert-base", num_labels=NUM_LABELS, problem_type="multi_label_classification"
)


def tokenize_function(examples):
    return tokenizer(
        examples["source_code"],
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
    )


# 3. Crear Datasets de HuggingFace
train_hf = HFDataset.from_dict(
    {
        "source_code": train_df_min_token["source_code"].tolist(),
        "labels": train_labels.tolist(),
    }
)

test_hf = HFDataset.from_dict(
    {
        "source_code": test_df_min_token["source_code"].tolist(),
        "labels": test_labels.tolist(),
    }
)

# Tokenizar y formatear
train_hf = train_hf.map(tokenize_function, batched=True).remove_columns(["source_code"])
test_hf = test_hf.map(tokenize_function, batched=True).remove_columns(["source_code"])
train_hf.set_format("torch")
test_hf.set_format("torch")

# 4. Cálculo de Pesos para el desbalanceo (pos_weight)
# Calculamos: (negativos / positivos) por cada clase
class_counts = train_labels.sum(axis=0)
pos_weights = (len(train_labels) - class_counts) / np.maximum(class_counts, 1)
# Aplicamos sqrt para que los pesos no sean tan extremos y desestabilicen el entrenamiento
pos_weights_tensor = torch.tensor(np.sqrt(pos_weights), dtype=torch.float).to(DEVICE)

print("\n⚖️ Pesos por clase (para balancear la pérdida):")
for cls, w in zip(VULN_CLASSES, pos_weights):
    print(f"  {cls:<22}: {np.sqrt(w):.2f}")


# 5. Métricas para Multi-label
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Aplicamos sigmoid para pasar de logits a probabilidades [0, 1]
    probs = 1 / (1 + np.exp(-logits))
    # Umbral de 0.5 para binarizar predicciones
    preds = (probs > 0.5).astype(float)

    return {
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
        "accuracy": accuracy_score(labels, preds),
    }


# 6. Trainer con Pérdida Pesada
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        # Usamos BCEWithLogitsLoss con los pesos calculados
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weights_tensor)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


# 7. Argumentos de Entrenamiento
training_args = TrainingArguments(
    output_dir="/kaggle/working/codebert_smartbugs_v2",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=True,  # Acelera entrenamiento en T4/P100
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=test_hf,
    compute_metrics=compute_metrics,
)

# --- INICIAR ENTRENAMIENTO ---
trainer.train()

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1107, training_loss=0.6058292406145904, metrics={'train_runtime': 1922.9905, 'train_samples_per_second': 18.414, 'train_steps_per_second': 0.576, 'total_flos': 9316750306507776.0, 'train_loss': 0.6058292406145904, 'epoch': 3.0})

In [40]:
import numpy as np
import pandas as pd
import torch

# 1. Obtener las predicciones del trainer
# Esto devuelve una tupla: (logits, labels, metrics)
output = trainer.predict(test_hf)

# 2. Procesar los Logits (activación Sigmoide)
# Aplicamos sigmoide para convertir logits en probabilidades entre 0 y 1
probs = 1 / (1 + np.exp(-output.predictions))

# 3. Binarizar (Umbral 0.5)
preds_binarized = (probs > 0.5).astype(int)

# 4. Valores reales (ya están en output.label_ids)
real_labels = output.label_ids.astype(int)

# 5. Crear el DataFrame
# VULN_CLASSES es la lista: ['Re-entrancy', 'Timestamp-Dependency', 'Unhandled-Exception', 'tx.origin', 'clean']
df_results = pd.DataFrame()

# Guardamos los valores reales y predichos columna por columna
for i, class_name in enumerate(VULN_CLASSES):
    df_results[f"real_{class_name}"] = real_labels[:, i]
    df_results[f"pred_{class_name}"] = preds_binarized[:, i]
    df_results[f"prob_{class_name}"] = probs[:, i]  # Guardamos la prob cruda por si querés ajustar el umbral después

# Opcional: Agregar el contract_id para trazabilidad (si lo tenés en el test_df_min_token)
if "contract_id" in test_df_min_token.columns:
    df_results["contract_id"] = test_df_min_token["contract_id"].values

# 6. Almacenar en archivo
output_path = "/kaggle/working/test_predictions_results.csv"
df_results.to_csv(output_path, index=False)

print(f"✅ Archivo de predicciones guardado en: {output_path}")

✅ Archivo de predicciones guardado en: /kaggle/working/test_predictions_results.csv


In [41]:
results = trainer.evaluate()

print("\n" + "=" * 50)
print("TEST RESULTS")
print("=" * 50)
print(f"  F1 Macro : {results['eval_f1_macro']:.4f}")
print(f"  F1 Micro : {results['eval_f1_micro']:.4f}")
print(f"  Recall   : {results['eval_recall']:.4f}")
print(f"  Accuracy : {results['eval_accuracy']:.4f}")
print("=" * 50)

# Guardar modelo y tokenizer
save_path = "/kaggle/working/codebert_smartbugs_weighted"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"\nModelo guardado en {save_path}")


TEST RESULTS
  F1 Macro : 0.2230
  F1 Micro : 0.2676


KeyError: 'eval_recall'

In [ ]:
!zip -r output.zip /kaggle/working/codebert_smartbugs

In [ ]:
def get_predictions(trainer, dataset, mlb, logits_raw=None):
    if logits_raw is None:
        output = trainer.predict(dataset)
        logits_raw = output.predictions
        labels = output.label_ids
    else:
        labels = np.array([dataset[i]["labels"].numpy() for i in range(len(dataset))])

    probs = torch.sigmoid(torch.tensor(logits_raw)).numpy()

    # Para test single-label: tomar la clase con mayor probabilidad
    pred_flat = np.argmax(probs, axis=1)
    true_flat = np.argmax(labels, axis=1)

    pred_names = [mlb.classes_[i] for i in pred_flat]
    true_names = [mlb.classes_[i] for i in true_flat]

    return {
        "pred_flat": pred_flat,
        "true_flat": true_flat,
        "pred_names": pred_names,
        "true_names": true_names,
        "probs": probs,
        "logits": logits_raw,
        "labels": labels,
        "classes": list(mlb.classes_),
    }


def plot_confusion_matrix(results, figsize=(10, 8)):
    """Muestra la matriz de confusión con porcentajes."""
    classes = results["classes"]
    cm = confusion_matrix(results["true_flat"], results["pred_flat"])

    # Normalizar por fila (recall por clase)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)

    fig, axes = plt.subplots(1, 2, figsize=(figsize[0] * 2, figsize[1]))

    # Absoluta
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes, ax=axes[0])
    axes[0].set_xlabel("Predicho")
    axes[0].set_ylabel("Real")
    axes[0].set_title("Matriz de Confusión (absoluta)")
    axes[0].tick_params(axis="x", rotation=45)
    axes[0].tick_params(axis="y", rotation=0)

    # Normalizada
    sns.heatmap(
        cm_norm,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        xticklabels=classes,
        yticklabels=classes,
        ax=axes[1],
    )
    axes[1].set_xlabel("Predicho")
    axes[1].set_ylabel("Real")
    axes[1].set_title("Matriz de Confusión (normalizada)")
    axes[1].tick_params(axis="x", rotation=45)
    axes[1].tick_params(axis="y", rotation=0)

    plt.tight_layout()
    plt.show()


def print_classification_report(results):
    """Imprime el reporte de clasificación por clase."""
    print(
        classification_report(
            results["true_names"],
            results["pred_names"],
            zero_division=0,
        )
    )


def show_samples(results, test_df, tokenizer, n=20, filter_wrong=False, filter_class=None, save_path=None):
    indices = list(range(len(results["true_names"])))

    if filter_wrong:
        indices = [i for i in indices if results["true_names"][i] != results["pred_names"][i]]

    if filter_class:
        indices = [i for i in indices if results["true_names"][i] == filter_class]

    indices = indices[:n]

    lines = []
    header = (
        f"Mostrando {len(indices)} muestras"
        + (" (solo errores)" if filter_wrong else "")
        + (f" (clase: {filter_class})" if filter_class else "")
    )
    lines.append(header)
    lines.append("=" * 100)

    for i in indices:
        code_full = str(test_df["source_code"].iloc[i])
        num_tokens = len(tokenizer.encode(code_full))
        truncado = " ⚠ TRUNCADO" if num_tokens > MAX_LEN else ""
        correct = "✓" if results["true_names"][i] == results["pred_names"][i] else "✗"

        top3_idx = np.argsort(results["probs"][i])[::-1][:3]
        top3 = [(results["classes"][j], results["probs"][i][j]) for j in top3_idx]
        top3_str = " | ".join([f"{name}: {prob:.3f}" for name, prob in top3])

        lines.append(f"[{correct}] Sample {i}")
        lines.append(f"  Real:      {results['true_names'][i]}")
        lines.append(f"  Predicho:  {results['pred_names'][i]}")
        lines.append(f"  Tokens:    {num_tokens} / {MAX_LEN}{truncado}")
        lines.append(f"  Top 3:     {top3_str}")
        lines.append("  Código completo:")
        lines.append("-" * 50)
        lines.append(code_full)
        lines.append("=" * 100)

    output = "\n".join(lines)
    print(output)

    if save_path:
        with open(save_path, "w") as f:
            f.write(output)
        print(f"\nGuardado en {save_path}")


def show_class_distribution(results):
    """Muestra distribución de clases reales vs predichas."""
    classes = results["classes"]
    true_counts = np.bincount(results["true_flat"], minlength=len(classes))
    pred_counts = np.bincount(results["pred_flat"], minlength=len(classes))

    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(classes))
    width = 0.35

    ax.bar(x - width / 2, true_counts, width, label="Real", color="steelblue")
    ax.bar(x + width / 2, pred_counts, width, label="Predicho", color="salmon")
    ax.set_xticks(x)
    ax.set_xticklabels(classes, rotation=45, ha="right")
    ax.set_ylabel("Cantidad")
    ax.set_title("Distribución de clases: Real vs Predicho")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Obtener predicciones
res = get_predictions(trainer, test_hf, mlb)

# Matriz de confusión
plot_confusion_matrix(res)

# Reporte por clase
print_classification_report(res)

# Distribución real vs predicho
show_class_distribution(res)

# Ver errores de una clase específica
show_samples(
    res,
    test_df,
    tokenizer,
    n=30,
    filter_wrong=True,
    save_path="/kaggle/working/errores_detalle.txt",
)

In [ ]:
# Distribución de clases en train
print("TRAIN:")
print(train_df["labels_list"].explode().value_counts())
print()

# Distribución de clases en test
print("TEST:")
print(test_df["labels_list"].explode().value_counts())

In [ ]:
def generate_report(
    trainer,
    test_df,
    test_hf,
    mlb,
    tokenizer,
    run_name="run",
    hyperparams=None,
    save_dir="/kaggle/working",
):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    prefix = f"{save_dir}/{run_name}_{timestamp}"

    res = get_predictions(trainer, test_hf, mlb)
    classes = res["classes"]  # <-- definir acá

    report_dict = classification_report(
        res["true_names"],
        res["pred_names"],
        zero_division=0,
        output_dict=True,
    )

    lines = []
    lines.append("=" * 80)
    lines.append(f"REPORTE DE ENTRENAMIENTO: {run_name}")
    lines.append(f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    lines.append("=" * 80)

    lines.append("\n--- HIPERPARÁMETROS ---")
    if hyperparams:
        for k, v in hyperparams.items():
            lines.append(f"  {k}: {v}")
    else:
        lines.append("  (no especificados)")

    lines.append("\n--- MÉTRICAS GENERALES ---")
    lines.append(f"  F1 Macro : {report_dict['macro avg']['f1-score']:.4f}")
    lines.append(f"  F1 Micro : {report_dict.get('weighted avg', {}).get('f1-score', 0):.4f}")
    lines.append(f"  Recall   : {report_dict['macro avg']['recall']:.4f}")
    lines.append(f"  Accuracy : {report_dict['accuracy']:.4f}")

    lines.append("\n--- MÉTRICAS POR CLASE ---")
    lines.append(f"  {'Clase':<25} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
    lines.append("  " + "-" * 65)
    for cls in mlb.classes_:
        if cls in report_dict:
            r = report_dict[cls]
            lines.append(
                f"  {cls:<25} {r['precision']:>10.4f} {r['recall']:>10.4f} {r['f1-score']:>10.4f} {r['support']:>10.0f}"
            )

    lines.append("\n--- DISTRIBUCIÓN TRAIN ---")
    train_counts = train_df["labels_list"].explode().value_counts()
    for cls, count in train_counts.items():
        lines.append(f"  {cls:<25} {count}")

    lines.append("\n--- DISTRIBUCIÓN TEST ---")
    test_counts = test_df["labels_list"].explode().value_counts()
    for cls, count in test_counts.items():
        lines.append(f"  {cls:<25} {count}")

    lines.append("\n--- MATRIZ DE CONFUSIÓN ---")
    cm = confusion_matrix(res["true_flat"], res["pred_flat"], labels=list(range(len(classes))))
    header = f"  {'':>22}" + "".join(f"{c[:12]:>13}" for c in classes)
    lines.append(header)
    for i, cls in enumerate(classes):
        row = f"  {cls:>22}" + "".join(f"{cm[i][j]:>13}" for j in range(len(classes)))
        lines.append(row)

    lines.append("\n--- HISTORIAL DE ENTRENAMIENTO ---")
    if trainer.state.log_history:
        for entry in trainer.state.log_history:
            if "loss" in entry or "eval_loss" in entry:
                lines.append(f"  {entry}")

    lines.append("\n\n" + "=" * 80)
    lines.append("DETALLE DE TODAS LAS PREDICCIONES (TEST)")
    lines.append("=" * 80)

    wrong_indices = [i for i in range(len(res["true_names"])) if res["true_names"][i] != res["pred_names"][i]]
    correct_indices = [i for i in range(len(res["true_names"])) if res["true_names"][i] == res["pred_names"][i]]

    lines.append(f"\nTotal: {len(res['true_names'])} samples")
    lines.append(f"  Correctos: {len(correct_indices)} ({len(correct_indices) / len(res['true_names']) * 100:.1f}%)")
    lines.append(f"  Errores:   {len(wrong_indices)} ({len(wrong_indices) / len(res['true_names']) * 100:.1f}%)")

    for i in range(len(res["true_names"])):
        code_full = str(test_df["source_code"].iloc[i])
        num_tokens = len(tokenizer.encode(code_full))

        top3_idx = np.argsort(res["probs"][i])[::-1][:3]
        top3 = [(classes[j], res["probs"][i][j]) for j in top3_idx]
        top3_str = " | ".join([f"{name}: {prob:.3f}" for name, prob in top3])

        is_correct = res["true_names"][i] == res["pred_names"][i]
        mark = "✓" if is_correct else "✗"

        lines.append(f"\n[{mark}] Sample {i}")
        lines.append(f"  Real:      {res['true_names'][i]}")
        lines.append(f"  Predicho:  {res['pred_names'][i]}")
        lines.append(f"  Tokens:    {num_tokens}")
        lines.append(f"  Top 3:     {top3_str}")
        lines.append("  Código:")
        lines.append("-" * 50)
        lines.append(code_full)
        lines.append("-" * 50)

    report_path = f"{prefix}_report.txt"
    with open(report_path, "w") as f:
        f.write("\n".join(lines))

    json_path = f"{prefix}_metrics.json"
    json_data = {
        "run_name": run_name,
        "timestamp": timestamp,
        "hyperparams": hyperparams or {},
        "metrics": {
            "f1_macro": report_dict["macro avg"]["f1-score"],
            "f1_micro": report_dict.get("weighted avg", {}).get("f1-score", 0),
            "recall": report_dict["macro avg"]["recall"],
            "accuracy": report_dict["accuracy"],
        },
        "per_class": {cls: report_dict.get(cls, {}) for cls in mlb.classes_},
    }
    with open(json_path, "w") as f:
        json.dump(json_data, f, indent=2)

    cm_path = f"{prefix}_confusion_matrix.png"
    cm = confusion_matrix(res["true_flat"], res["pred_flat"], labels=list(range(len(classes))))
    cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes, ax=axes[0])
    axes[0].set_xlabel("Predicho")
    axes[0].set_ylabel("Real")
    axes[0].set_title("Matriz de Confusión (absoluta)")
    axes[0].tick_params(axis="x", rotation=45)

    sns.heatmap(
        cm_norm,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        xticklabels=classes,
        yticklabels=classes,
        ax=axes[1],
    )
    axes[1].set_xlabel("Predicho")
    axes[1].set_ylabel("Real")
    axes[1].set_title("Matriz de Confusión (normalizada)")
    axes[1].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.show()

    print("\nArchivos guardados:")
    print(f"  Reporte:  {report_path}")
    print(f"  Métricas: {json_path}")
    print(f"  Matriz:   {cm_path}")

    return res

In [ ]:
res = generate_report(
    trainer,
    test_df,
    test_hf,
    mlb,
    tokenizer,
    run_name="v2_con_pesos_lr0_0001",
    hyperparams={
        "model": "microsoft/codebert-base",
        "max_len": MAX_LEN,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "warmup_steps": 100,
        "weight_decay": 0.01,
        "pos_weight": "ninguno",
        "fp16": True,
        "notas": "Sin weighted trainer, menos lr",
    },
)